# Day 37 — MLflow tracking + preprocessing contract tests

Status: COMPLETE. Experiment `pneumonia_xray` holds all three runs
(params + 6 metrics each, tagged backfilled); preprocessing pins exact
shape/dtype/normalization behavior. 15/15 tests pass.

In [1]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import mlflow

mlflow.set_tracking_uri("../mlruns")
exp = mlflow.get_experiment_by_name("pneumonia_xray")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
for _, r in runs.sort_values("metrics.roc_auc").iterrows():
    print(f"{r['tags.mlflow.runName']:<18}: "
          f"ROC-AUC {r['metrics.roc_auc']} | PR-AUC {r['metrics.pr_auc']}")
print("browse: .venv/bin/mlflow ui --backend-store-uri ./mlruns")

cnn_baseline      : ROC-AUC 0.8550 | PR-AUC 0.9055
mobilenet_frozen  : ROC-AUC 0.9584 | PR-AUC 0.9735
mobilenet_finetuned: ROC-AUC 0.9622 | PR-AUC 0.9714
browse: .venv/bin/mlflow ui --backend-store-uri ./mlruns


## Findings

1. **Tracking is a progression table**: the three runs read as the project
   story (0.855 → 0.958 → 0.962) — exactly what experiment tracking is for.
2. **Backfill, honestly labeled.** Retraining 3 models (~1.5h CPU) to regenerate
   identical numbers would be theater; `src/backfill_mlflow.py` registers the
   recorded results tagged `backfilled=true`, and the train scripts remain the
   source of truth for future live-logged runs.
3. **Preprocessing tests assert values, not vibes**: solid-gray-128 maps to
   exact per-channel normalized constants — a wrong mean/std or channel swap
   fails loudly instead of silently shifting every prediction.